# AD5780 / Keysight driver tests â€” no hardware
Run all cells, or rerun an individual test after the setup cell. Every test's inputs and assertions are here. Fake transports do not contact instruments. An assertion or exception produces a notebook traceback.


In [ ]:
from pathlib import Path
import sys, socket, math, types
from contextlib import contextmanager
from unittest.mock import patch
root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "malab" / "instruments" / "ad5780.py").exists())
sys.path.insert(0, str(root))
import malab
assert Path(malab.__file__).resolve().parent == root / "malab", "Restart kernel: another malab is already imported"
from malab.instruments import AD5780, Keysight34461A, DACBenchControls
from malab.instruments.ad5780 import V_PER_BIT, MAX_CODE, voltage_to_code, code_to_voltage
from malab.instruments._line_socket import (
    InstrumentTimeout, InstrumentConnectionError, ProtocolError, CommandError)
BANNER = b"Welcome to the Telnet server!\r\nType HELP for instructions.\r\n"
IDENTITY = b"Keysight Technologies,34461A,MY60041761,A.03.03\n"

class FakeSocket:
    def __init__(self, chunks):
        self.chunks, self.sent, self.closed = list(chunks), [], False
    def settimeout(self, value):
        assert value > 0
    def sendall(self, data):
        self.sent.append(data)
    def recv(self, size):
        if not self.chunks:
            raise socket.timeout("No more fake replies")
        item = self.chunks.pop(0)
        if isinstance(item, Exception):
            raise item
        if len(item) > size:
            self.chunks.insert(0, item[size:])
        return item[:size]
    def close(self):
        self.closed = True

@contextmanager
def raises(kind):
    try:
        yield
    except kind as exc:
        print(type(exc).__name__ + ": " + str(exc))
    else:
        raise AssertionError("Expected " + kind.__name__)

@contextmanager
def fake_device(cls, replies, **kwargs):
    if cls is AD5780:
        kwargs.setdefault("use_calibration", False)  # Nominal transport fixtures; device calibration is tested below.
    transport = FakeSocket(replies)
    with patch("malab.instruments._line_socket.socket.create_connection", return_value=transport):
        device = cls(**kwargs)
        try:
            yield device, transport
        finally:
            device.close()
print("Imports:", malab.__file__)


## 1. Fragmented welcome and replies; coalesced lines
One TCP receive may contain a partial line or several replies. No welcome text may leak into ID/READ.


In [ ]:
with fake_device(AD5780, [BANNER[:11], BANNER[11:] + b"AD5", b"780\r\n131072\r\n"]) as (dac, wire):
    assert dac.get_id() == "AD5780"
    assert dac.get_voltage(1) == 0
    assert wire.sent == [b"ID\n", b"READ 1\n"]


## 2. Exact offset-binary conversion and endpoint quantization


In [ ]:
assert voltage_to_code(-10) == 0
assert voltage_to_code(0) == 131072
assert voltage_to_code(10) == 262143
assert code_to_voltage(0) == -10
assert code_to_voltage(131072) == 0
assert code_to_voltage(262143) == 10 - V_PER_BIT
for voltage in (-0.05, -0.001, 0, 0.001, 0.05):
    assert abs(code_to_voltage(voltage_to_code(voltage)) - voltage) <= V_PER_BIT / 2


## 3. Invalid inputs never send a command


In [ ]:
with fake_device(AD5780, [BANNER]) as (dac, wire):
    for channel in (0, 5, -1, True, 1.5, "1"):
        with raises(ValueError):
            dac.set_voltage(channel, 0)
    for voltage in (-10.1, 10.1, float("nan"), float("inf"), None, "0", True):
        with raises(ValueError):
            dac.set_voltage(1, voltage)
    for speed in (0, -1, True, float("inf"), None, "0.1", 1e-300):
        with raises(ValueError):
            dac.ramp(1, 0, speed)
    for command in ("", " ", "ID\nSET 1 0", "ID\x00"):
        with raises(ValueError):
            dac.query(command)
    assert wire.sent == []


## 4. SET acknowledgment and READ voltage semantics


In [ ]:
with fake_device(AD5780, [BANNER, b"SET command received with parameters: 1 131727\n131727\n"]) as (dac, wire):
    quantized = dac.set_voltage(1, 0.05)
    assert quantized == code_to_voltage(131727)
    assert dac.get_voltage(1) == quantized
    assert wire.sent == [b"SET 1 131727\n", b"READ 1\n"]


## 5. Device rejection is explicit, without damaging a correctly framed session


In [ ]:
with fake_device(AD5780, [BANNER, b"ERR! DAC busy.\nAD5780\n"]) as (dac, wire):
    with raises(CommandError):
        dac.set_voltage(1, 0)
    assert dac.connected
    assert dac.get_id() == "AD5780"
    assert len(wire.sent) == 2


## 6. Timeout/EOF close the socket; SET is never replayed


In [ ]:
for failure, expected in [(socket.timeout("slow"), InstrumentTimeout),
                          (b"", InstrumentConnectionError), (OSError("link lost"), InstrumentConnectionError)]:
    with fake_device(AD5780, [BANNER, failure]) as (dac, wire):
        with raises(expected):
            dac.set_voltage(1, 0)
        assert not dac.connected and wire.closed
        assert wire.sent == [b"SET 1 131072\n"]
        with raises(InstrumentConnectionError):
            dac.get_id()
        assert len(wire.sent) == 1


## 7. Invalid/oversized replies close the session


In [ ]:
for reply in (b"262144\n", b"-1\n", b"None\n", b"\xff\n", b"9" * 9000,
              b"#Help:\nCommands:\n"):
    with fake_device(AD5780, [BANNER, reply]) as (dac, wire):
        with raises(ProtocolError):
            dac.get_code(1)
        assert not dac.connected


## 8. Disabled clients perform no network I/O; connection failures are explicit


In [ ]:
with patch("malab.instruments._line_socket.socket.create_connection") as create:
    dac = AD5780(address='192.0.2.10',enabled=False, recv_length=1024)
    with raises(InstrumentConnectionError):
        dac.get_id()
    create.assert_not_called()
for failure, expected in [(socket.timeout(), InstrumentTimeout), (OSError("refused"), InstrumentConnectionError)]:
    with patch("malab.instruments._line_socket.socket.create_connection", side_effect=failure):
        with raises(expected):
            AD5780(address='192.0.2.10',)
with patch("malab.instruments._line_socket.socket.create_connection",
           return_value=FakeSocket([b"ERR! Server busy: only one connection allowed.\n"])):
    with raises(CommandError):
        AD5780(address='192.0.2.10',)


## 9. Ten-code ramp steps: a tenfold speed change gives a tenfold delay change
The firmware delay is **microseconds**, not seconds or milliseconds.


In [ ]:
for speed, delay in [(0.001, 762940), (0.01, 76294)]:
    target = voltage_to_code(0.05)
    ack = f"RAMP ACCEPTED 1 {target} 10 {delay}\n".encode()
    with fake_device(AD5780, [BANNER, ack]) as (dac, wire):
        status = dac.ramp(1, 0.05, speed, wait=False)
        assert status.state == "ACCEPTED"
        assert wire.sent == [f"RAMP 1 {target} 10 {delay}\n".encode()]
        assert delay * 1e-6 >= 10 * V_PER_BIT / speed


## 10. Ramp completion comes from STATUS, not merely acceptance


In [ ]:
target = voltage_to_code(0.05)
replies = f"RAMP ACCEPTED 1 {target} 10 76294\nRAMP RUNNING 1 {target}\nRAMP COMPLETE 1 {target}\n".encode()
with fake_device(AD5780, [BANNER, replies]) as (dac, wire):
    with patch("malab.instruments.ad5780.time.sleep"):
        status = dac.ramp(1, 0.05, 0.01)
    assert status.state == "COMPLETE"
    assert wire.sent[-2:] == [b"STATUS\n", b"STATUS\n"]


## 11. Aborted/failed/replaced ramps must not report success


In [ ]:
target = voltage_to_code(0.05)
for status_line in (f"RAMP ABORTED 1 {target}", f"RAMP FAILED 1 {target}",
                    f"RAMP COMPLETE 2 {target}", "RAMP nonsense 1 0"):
    reply = f"RAMP ACCEPTED 1 {target} 10 76294\n{status_line}\n".encode()
    with fake_device(AD5780, [BANNER, reply]) as (dac, wire):
        with raises(ProtocolError if "nonsense" in status_line else CommandError):
            dac.ramp(1, 0.05, 0.01)
        assert sum(s.startswith(b"RAMP ") for s in wire.sent) == 1


## 12. Overall wait deadline does not silently change outputs or replay the ramp
The caller must choose whether to close the connection to abort a still-running ramp.


In [ ]:
target = voltage_to_code(0.05)
with fake_device(AD5780, [BANNER, f"RAMP ACCEPTED 1 {target} 10 76294\nRAMP RUNNING 1 {target}\n".encode()]) as (dac, wire):
    # Advance only the ramp's clock; transport I/O uses its own real deadline.
    clock = [0.0]
    fake_time = types.SimpleNamespace(monotonic=lambda: clock[0],
        sleep=lambda seconds: clock.__setitem__(0, clock[0]+seconds))
    with patch("malab.instruments.ad5780.time", fake_time), raises(InstrumentTimeout):
        dac.ramp(1, 0.05, 0.01, timeout=1, poll_interval=1)
    assert dac.connected
    assert sum(s.startswith(b"RAMP ") for s in wire.sent) == 1
    dac.abort_ramp()
    assert not dac.connected


## 13. INIT is explicit and validates its reply


In [ ]:
with fake_device(AD5780, [BANNER, b"Initialized DAC 1; active outputs preserved.\n"]) as (dac, wire):
    assert wire.sent == []  # Constructor never initializes outputs.
    assert "preserved" in dac.initialize(1)
    assert wire.sent == [b"INIT 1\n"]


## 14. Meter identification, DC setup, measurement, and error checking


In [ ]:
with fake_device(Keysight34461A, [IDENTITY, b'0,"No error"\n1\n0,"No error"\n+4.999900E-02\n']) as (meter, wire):
    with raises(CommandError):
        meter.read_voltage()
    meter.configure_dc_voltage(10, nplc=1)
    assert meter.read_voltage() == 0.049999
    assert wire.sent == [b"*IDN?\n", b"SYST:ERR?\n", b"CONF:VOLT:DC 10\n",
        b"VOLT:DC:NPLC 1\n", b"TRIG:SOUR IMM\n", b"TRIG:COUN 1\n", b"SAMP:COUN 1\n",
        b"*OPC?\n", b"SYST:ERR?\n", b"READ?\n"]


## 15. Wrong meter, SCPI errors and invalid measurements are explicit failures


In [ ]:
with raises(ProtocolError):
    with fake_device(Keysight34461A, [b"Vendor,WrongModel,123,1\n"]):
        pass
with fake_device(Keysight34461A, [IDENTITY, b'-113,"Undefined header"\n0,"No error"\n']) as (meter, wire):
    with raises(CommandError):
        meter.configure_dc_voltage()
    assert not any(s.startswith(b"CONF:") for s in wire.sent)
for value in (b"9.9E37\n", b"NaN\n", b"invalid\n"):
    with fake_device(Keysight34461A, [IDENTITY, b'0,"No error"\n1\n0,"No error"\n', value]) as (meter, wire):
        meter.configure_dc_voltage()
        with raises(ProtocolError):
            meter.read_voltage()


## 16. Meter settings are validated before I/O


In [ ]:
with fake_device(Keysight34461A, [IDENTITY]) as (meter, wire):
    for settings in [(2, 1), (10, 0.1), (True, 1), (10, False)]:
        with raises(ValueError):
            meter.configure_dc_voltage(*settings)
    assert wire.sent == [b"*IDN?\n"]


## 17. USB action replies are distinguished from ordinary diagnostics


In [ ]:
class FakeSerial:
    def __init__(self, replies):
        self.data = bytearray(replies)
        self.sent = []
        self.is_open = True
    @property
    def in_waiting(self):
        return len(self.data)
    def write(self, data):
        self.sent.append(data)
        return len(data)
    def read(self, size):
        result = bytes(self.data[:size])
        del self.data[:size]
        return result
    def close(self):
        self.is_open = False
usb = FakeSerial(b"Beginning SPI bus...\nBENCH STATUS READY\nNetwork ready.\nBENCH OK NETWORK\nBENCH OK SAFEZERO\nBENCH STATUS LATCHED\n")
with patch.dict(sys.modules, {"serial": types.SimpleNamespace(Serial=lambda *a, **k: usb)}):
    with DACBenchControls("FAKE") as controls:
        controls.network_reset()
        controls.safe_zero()
        assert controls.status() == "LATCHED"
assert usb.sent == [b"BENCH STATUS\n", b"BENCH NETWORK\n", b"BENCH SAFEZERO\n", b"BENCH STATUS\n"]
assert not usb.is_open


## 18. Missing USB response cannot silently count as a button action


In [ ]:
usb = FakeSerial(b"BENCH STATUS READY\n")
with patch.dict(sys.modules, {"serial": types.SimpleNamespace(Serial=lambda *a, **k: usb)}):
    controls = DACBenchControls("FAKE", timeout=0.01)
    with raises(InstrumentTimeout):
        controls.network_reset()
    assert not usb.is_open
    assert usb.sent.count(b"BENCH NETWORK\n") == 1
    with raises(InstrumentConnectionError):
        controls.status()


## 19. Device calibration drives SET, GET and an explicit achievable range


In [ ]:
cal_line = b"CAL 1 -9992845256 76283553 130996 1\n"
with fake_device(AD5780, [BANNER, cal_line + b"SET command received with parameters: 1 130996\n130996\n"], use_calibration=True) as (dac, wire):
    zero = dac.set_voltage(1, 0)
    assert abs(zero) < V_PER_BIT/2
    assert dac.get_voltage(1) == zero
    lo,hi = dac.get_voltage_limits(1)
    assert -9.993 < lo < -9.992 and hi == 10
    with raises(ValueError):
        dac.set_voltage(1, -10)
    assert wire.sent == [b"CAL 1\n", b"SET 1 130996\n", b"READ 1\n"]


## 20. Calibrated V/s conversion uses the measured volts per code


In [ ]:
target = 133618
# Compute independently from the documented integer calibration fields.
target = round((0.2 + 9.992845256) / 0.000076283553)
delay = math.ceil(10*0.000076283553/0.1*1e6)
with fake_device(AD5780, [BANNER, cal_line + f"RAMP ACCEPTED 1 {target} 10 {delay}\n".encode()], use_calibration=True) as (dac, wire):
    status = dac.ramp(1, 0.2, 0.1, wait=False)
    assert status.target_code == target
    assert wire.sent == [b"CAL 1\n", f"RAMP 1 {target} 10 {delay}\n".encode()]


## 21. Malformed calibration cannot become a voltage or output command


In [ ]:
for bad in (b"CAL 2 -9992845256 76283553 130996 1\n", b"CAL 1 -9992845256 0 130996 1\n",
            b"CAL 1 -9992845256 76283553 0 1\n", b"nonsense\n"):
    with fake_device(AD5780, [BANNER, bad], use_calibration=True) as (dac, wire):
        with raises(ProtocolError):
            dac.set_voltage(1, 0)
        assert wire.sent == [b"CAL 1\n"] and not dac.connected


## 22. Public AD5780 safe-zero works with Ethernet disabled


In [ ]:
usb = FakeSerial(b"BENCH STATUS READY\nBENCH OK SAFEZERO\nBENCH STATUS ACTIVE\nBENCH STATUS LATCHED\n")
with patch.dict(sys.modules, {"serial": types.SimpleNamespace(Serial=lambda *a, **k: usb)}):
    with patch("malab.instruments._line_socket.socket.create_connection") as create:
        dac=AD5780(address='192.0.2.10',enabled=False,bench_port="FAKE")
        try:
            assert dac.safe_zero(wait=True,poll_interval=0.001)=="LATCHED"
            create.assert_not_called()
        finally:
            dac.close()
assert not usb.is_open
assert usb.sent.count(b"BENCH SAFEZERO\n")==1


## 23. Public safe-zero must raise when firmware reports FAILED


In [ ]:
usb = FakeSerial(b"BENCH STATUS READY\nBENCH OK SAFEZERO\nBENCH STATUS FAILED\n")
with patch.dict(sys.modules, {"serial": types.SimpleNamespace(Serial=lambda *a, **k: usb)}):
    dac=AD5780(address='192.0.2.10',enabled=False,bench_port="FAKE")
    try:
        with raises(CommandError):
            dac.safe_zero(wait=True)
    finally:
        dac.close()
assert usb.sent.count(b"BENCH SAFEZERO\n")==1


## 24. Public NETWORK invalidates TCP and keeps independent USB status usable


In [ ]:
usb = FakeSerial(b"BENCH STATUS READY\nBENCH OK NETWORK\nBENCH STATUS READY\n")
with patch.dict(sys.modules, {"serial": types.SimpleNamespace(Serial=lambda *a, **k: usb)}):
    with fake_device(AD5780, [BANNER], bench_port="FAKE") as (dac, wire):
        assert dac.network_reset()=="BENCH OK NETWORK"
        assert not dac.connected and wire.closed and usb.is_open
        assert dac.get_recovery_status()=="READY"
assert not usb.is_open
assert usb.sent.count(b"BENCH NETWORK\n")==1


## 25. Public fault injection is explicit, validated and never changes outputs remotely


In [ ]:
usb=FakeSerial(b"BENCH STATUS READY\nBENCH OK FAULT READBACK\nBENCH OK FAULT CLEAR\nBENCH OK FAULT NETWORK\n")
with patch.dict(sys.modules, {"serial": types.SimpleNamespace(Serial=lambda *a, **k: usb)}):
    dac=AD5780(address='192.0.2.10',enabled=False,bench_port="FAKE")
    try:
        for kind in ('readback','clear','network'):
            assert dac.bench_inject_fault(kind)=='BENCH OK FAULT '+kind.upper()
        with raises(ValueError): dac.bench_inject_fault('arbitrary')
    finally: dac.close()
assert usb.sent==[b"BENCH STATUS\n",b"BENCH FAULT READBACK\n",b"BENCH FAULT CLEAR\n",b"BENCH FAULT NETWORK\n"]
with fake_device(AD5780,[BANNER,b"131072\n"]) as (dac,wire):
    dac.write('READ ',terminate=False)
    dac.write('1')
    assert dac.read()=='131072'
    assert wire.sent==[b"READ ",b"1\n"]
    with raises(ValueError): dac.write('READ 1',terminate=1)



## 26 — Live telemetry distinguishes checked reads, writes, and unknown values

In [ ]:
line=b"VALUES 130996 1 131072 2 0 0 262143 1\n"
with fake_device(AD5780,[BANNER,line]) as (dac,wire):
    values=dac.get_output_codes()
    assert [v['code'] for v in values]==[130996,131072,None,262143]
    assert [v['source'] for v in values]==['readback','write','unknown','readback']
    assert wire.sent==[b"VALUES\n"]
for line in [b"VALUES 0 3 0 0 0 0 0 0\n",b"VALUES 262144 1 0 0 0 0 0 0\n",b"VALUES 0 0\n"]:
    with fake_device(AD5780,[BANNER,line]) as (dac,wire):
        with raises(ProtocolError):dac.get_output_codes()
        assert not dac.connected
usb=FakeSerial(b"BENCH STATUS READY\nBENCH VALUES 130996 2 131072 2 131072 2 131072 2\n")
with patch.dict(sys.modules,{'serial':types.SimpleNamespace(Serial=lambda *a,**k:usb)}):
    with patch('malab.instruments._line_socket.socket.create_connection') as create:
        dac=AD5780(address='192.0.2.10',enabled=False,bench_port='FAKE')
        try:
            assert dac.get_output_codes(via_usb=True)[0]['code']==130996
            create.assert_not_called()
        finally:dac.close()
assert usb.sent==[b"BENCH STATUS\n",b"BENCH VALUES\n"]



## 27 — Ethernet disconnect, abort, and transport errors preserve USB recovery

In [ ]:
for operation in ('disconnect','abort','timeout','protocol'):
    usb=FakeSerial(b"BENCH STATUS READY\n"*3)
    replies=[BANNER]
    if operation=='timeout':replies.append(socket.timeout('link unavailable'))
    if operation=='protocol':replies.append(b'Unexpected ID\n')
    with patch.dict(sys.modules,{'serial':types.SimpleNamespace(Serial=lambda *a,**k:usb)}):
        with fake_device(AD5780,replies,bench_port='FAKE') as (dac,wire):
            assert dac.get_recovery_status()=='READY'
            if operation=='disconnect':dac.disconnect()
            elif operation=='abort':dac.abort_ramp()
            else:
                with raises(InstrumentTimeout if operation=='timeout' else ProtocolError):dac.get_id()
            assert not dac.connected and wire.closed
            assert usb.is_open, 'Ethernet operations must not close independent USB'
            assert dac.get_recovery_status()=='READY'
        assert not usb.is_open, 'Explicit full close still cleans up USB'
print('All 27 offline test groups passed.')
